# 02 – Data Cleaning
Cleans the PHS monthly A&E activity and "When" datasets based on findings from
`01_data_exploration.ipynb`. Outputs are saved to `data/processed/`.

In [60]:
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")

activity_file = next(RAW.glob("monthly_ae_activity*.csv"))
when_file = next(RAW.glob("*when*.csv"))

activity = pd.read_csv(activity_file)
when = pd.read_csv(when_file)

- Converting Month column which is in number format to a date column

In [61]:
activity["Month"] = pd.to_datetime(activity["Month"].astype(str), format="%Y%m")
when["Month"] = pd.to_datetime(when["Month"].astype(str),format="%Y%m")

In [62]:
print(activity["Month"].dtype, activity["Month"].min(), activity["Month"].max())
print(when["Month"].dtype, when["Month"].min(), when["Month"].max())

datetime64[us] 2007-07-01 00:00:00 2026-08-01 00:00:00
datetime64[us] 2018-01-01 00:00:00 2026-08-01 00:00:00


## 3. Filter rows
- **Date range:** keep January 2018 onwards to match the project scope and the period
  covered by the "When" file.
- **Attendance category:** keep only "All" (planned + unplanned) to avoid double counting.
  This matches the official 4-hour standard, which includes planned attendances since
  February 2025. Before December 2020, "All" is identical to "Unplanned".

In [63]:
print("Before : " , activity.shape)

activity = activity[activity["Month"] >= "2018-01-01"]
activity = activity[activity["AttendanceCategory"] == "All"]

print("After : ", activity.shape )

print(activity["AttendanceCategory"].unique())

Before :  (40066, 26)
After :  (7211, 26)
<StringArray>
['All']
Length: 1, dtype: str


## 4. Drop unneeded columns
- `Country`: single value (Scotland), so it carries no information.
- `AttendanceCategory`: only "All" remains after filtering.
- `QF` columns: contain only "z" (not applicable), which duplicates what missing
  Episode values already show.

In [64]:
qf_cols = [cols for cols in activity.columns if cols.endswith("QF")]
print(qf_cols)

activity = activity.drop(columns=["Country","AttendanceCategory"] + qf_cols)
when = when.drop(columns=["Country"])

print(activity.shape, when.shape)
print(activity.columns.tolist())

['NumberOfAttendancesEpisodeQF', 'NumberWithin4HoursEpisodeQF', 'NumberOver4HoursEpisodeQF', 'PercentageWithin4HoursEpisodeQF', 'NumberOver8HoursEpisodeQF', 'PercentageOver8HoursEpisodeQF', 'NumberOver12HoursEpisodeQF', 'PercentageOver12HoursEpisodeQF']
(7211, 16) (640703, 9)
['Month', 'HBT', 'TreatmentLocation', 'DepartmentType', 'NumberOfAttendancesAll', 'NumberWithin4HoursAll', 'NumberOver4HoursAll', 'PercentageWithin4HoursAll', 'NumberOfAttendancesEpisode', 'NumberWithin4HoursEpisode', 'NumberOver4HoursEpisode', 'PercentageWithin4HoursEpisode', 'NumberOver8HoursEpisode', 'PercentageOver8HoursEpisode', 'NumberOver12HoursEpisode', 'PercentageOver12HoursEpisode']


## 5. Add health board and hospital names
Health boards (`HBT`) and hospitals (`TreatmentLocation`) are stored as codes. PHS lookup
files are used to add readable names:
- `hb14_hb19.csv`: health board codes and names (covers 2014 and 2019 codes)
- `hospitals.csv`: current NHS hospitals in Scotland

First, inspect the lookup columns to identify the code and name fields.

In [65]:
hb_lookup = pd.read_csv(RAW/"hb14_hb19.csv")
hosp_lookup = pd.read_csv(RAW/"hospitals.csv")

print(hb_lookup.columns.tolist())
print(hosp_lookup.columns.tolist())

hb_lookup.head()

['HB', 'HBName', 'HBDateEnacted', 'HBDateArchived', 'Country']
['HospitalCode', 'HospitalName', 'AddressLine1', 'AddressLine2', 'AddressLine2QF', 'AddressLine3', 'AddressLine3QF', 'AddressLine4', 'AddressLine4QF', 'Postcode', 'HealthBoard', 'HSCP', 'CouncilArea', 'IntermediateZone', 'DataZone']


,HB,HBName,HBDateEnacted,HBDateArchived,Country
0,S08000015,NHS Ayrshire and Arran,20140401,NaN,S92000003
1,S08000016,NHS Borders,20140401,NaN,S92000003
2,S08000017,NHS Dumfries and Galloway,20140401,NaN,S92000003
3,S08000018,NHS Fife,20140401,20180201.0,S92000003
4,S08000019,NHS Forth Valley,20140401,NaN,S92000003


### 5.0 Join names onto the data
Join the lookup tables to both `activity` and `when` (similar to a VLOOKUP / SQL LEFT JOIN):
- `HBT` → `HBName` using `hb14_hb19.csv`
- `TreatmentLocation` → `HospitalName` using `hospitals.csv`

Settings used to keep the data safe:
- `how="left"` keeps every row, even if a code has no match. An inner join was
  found to silently drop 301 activity rows and 1,839 "When" rows from unmatched sites.
- `validate="many_to_one"` checks each code appears only once in the lookup, so no
  rows are duplicated and attendance totals are not inflated.

**Check:** row counts before and after the join must be identical.
**Result:** 7,211 activity rows and 640,703 "When" rows before and after.

In [66]:
hb_names = hb_lookup[["HB","HBName"]]
hosp_names = hosp_lookup[["HospitalCode","HospitalName"]]

rows_before = len(activity), len(when)

activity =(activity
           .merge(hb_names, left_on="HBT", right_on="HB", how="left", validate="many_to_one")
           .merge(hosp_names, left_on="TreatmentLocation", right_on="HospitalCode", how="left", validate="many_to_one")
           .drop(columns=["HB","HospitalCode"]))

when = (when
        .merge(hb_names, left_on="HBT", right_on="HB", how="left", validate="many_to_one")
        .merge(hosp_names, left_on="TreatmentLocation", right_on="HospitalCode", how="left", validate="many_to_one")
        .drop(columns=["HB", "HospitalCode"]))

print("Rows before:", rows_before, "| Rows after:", (len(activity), len(when)))

Rows before: (7211, 640703) | Rows after: (7211, 640703)


### 5.1 Check for unmatched codes
After the left joins, any code not found in a lookup file has an empty name.
Count missing board and hospital names, then list the unmatched hospital codes.

**Result:** all 14 health boards matched. 9 treatment locations (3 in NHS Grampian,
4 in NHS Highland, 2 in NHS Tayside) have no match in the current hospitals lookup,
affecting 301 of 7,211 activity rows and 1,839 of 640,703 "When" rows.
These are likely sites that have closed or changed code since 2018, as the lookup
only lists hospitals that are currently open.

In [67]:
print("Missing Hospital Boards in activity :", activity["HBName"].isna().sum())
print("Missing Hospital Names in activity : ", activity["HospitalName"].isna().sum(), " of " ,len(activity))
print("Missing Hospital Names in when : ", when["HospitalName"].isna().sum(), " of ", len(when))

unmatched = (activity
             .loc[activity["HospitalName"].isna(), ["TreatmentLocation","HBName"]].drop_duplicates())

unmatched


Missing Hospital Boards in activity : 0
Missing Hospital Names in activity :  301  of  7211
Missing Hospital Names in when :  1839  of  640703


,TreatmentLocation,HBName
25,N332H,NHS Grampian
35,N451H,NHS Grampian
37,N683C,NHS Grampian
57,H210H,NHS Highland
59,H214H,NHS Highland
64,H227B,NHS Highland
83,T305H,NHS Tayside
87,T357C,NHS Tayside
1861,H221B,NHS Highland


### 5.2 Investigate unmatched sites
Before deciding how to handle the unmatched sites, summarise each one:
department type, first and last month of data, number of months reported, and
total attendances. Also calculate their share of all attendances to judge
whether they materially affect the results.

In [68]:
unmatched_summary = (activity[activity["HospitalName"].isna()]
                     .groupby(["TreatmentLocation", "HBName", "DepartmentType"])
                     .agg(first_month = ("Month","min"),
                          last_month = ("Month","max"),
                          months = ("Month", "count"),
                          total_attendances= ("NumberOfAttendancesAll","sum"))
                     .sort_values("total_attendances",ascending =  False))

share = (activity.loc[activity["HospitalName"].isna(), "NumberOfAttendancesAll"].sum() /
                activity["NumberOfAttendancesAll"].sum())

print(f"Unmatched Sites :  {share: .2%} of all attendances ")

unmatched_summary

Unmatched Sites :   0.32% of all attendances 


,,,first_month,last_month,months,total_attendances
TreatmentLocation,HBName,DepartmentType,,,,
H214H,NHS Highland,Type 3,2018-01-01,2022-03-01,51,18484
H227B,NHS Highland,Type 3,2018-01-01,2026-08-01,104,13349
T357C,NHS Tayside,Type 3,2018-01-01,2020-03-01,27,6551
T305H,NHS Tayside,Type 3,2018-01-01,2019-06-01,18,2008
N683C,NHS Grampian,Type 3,2018-01-01,2020-03-01,27,1188
H210H,NHS Highland,Type 3,2018-01-01,2019-08-01,20,1134
N332H,NHS Grampian,Type 3,2018-01-01,2020-03-01,27,838
N451H,NHS Grampian,Type 3,2018-01-01,2019-11-01,23,363
H221B,NHS Highland,Type 3,2019-09-01,2019-12-01,4,178


**Result:** the 9 unmatched sites account for 0.32% of all attendances. All are Type 3
(minor injury units / small community sites). Eight stopped reporting between 2019 and
2022 (three in March 2020), consistent with closures or service changes. One site
(H227B) is still reporting but is not listed in the current hospitals file.

**Decision:** keep these rows so totals remain consistent with published PHS figures,
and label them as "Unlisted site (code)" rather than deleting them or adding names manually.

In [69]:
activity["HospitalName"] = activity["HospitalName"].fillna("Unlisted site (" + activity["TreatmentLocation"] + ") ")
when["HospitalName"] = when["HospitalName"].fillna("Unlisted site (" + when["TreatmentLocation"] + ")")

print(activity["HospitalName"].isna().sum() , when["HospitalName"].isna().sum())

0 0


## 6. Clean the "When" table
- Add `HourStart` (0–23) extracted from the `Hour` text (e.g. "00:00 to 00:59" → 0).
- Add `DayNumber` (Monday = 1 … Sunday = 7) so days sort in calendar order.
Numeric sort columns are used instead of pandas categories because they are kept when
saving to CSV and can be used directly in SQL and Power BI ("Sort by column").

In [70]:
when["HourStart"] = when["Hour"].str[:2].astype(int)

day_order = {"Monday" : 1, "Tuesday" : 2, "Wednesday" : 3, "Thursday" : 4, "Friday" : 5,
             "Saturday" : 6, "Sunday" : 7}

when["DayNumber"] = when["Day"].map(day_order)

print(when[["Hour","HourStart"]].drop_duplicates().sort_values("HourStart").head())
print(when[["Day","DayNumber"]].drop_duplicates().sort_values("DayNumber"))
print("Missing Day NUmber : ", when["DayNumber"].isna().sum())

             Hour  HourStart
0  00:00 to 00:59          0
1  01:00 to 01:59          1
2  02:00 to 02:59          2
3  03:00 to 03:59          3
4  04:00 to 04:59          4
           Day  DayNumber
24      Monday          1
120    Tuesday          2
144  Wednesday          3
96    Thursday          4
0       Friday          5
48    Saturday          6
72      Sunday          7
Missing Day NUmber :  0


**Result:** 24 hours mapped to 0–23 and 7 days mapped to 1–7 (Monday–Sunday), with no unmatched values.

## 7. Validation checks
Confirm the cleaned data is internally consistent before saving:
1. Within 4 hours + over 4 hours = total attendances, for every row.
2. Recalculated 4-hour percentage matches the published `PercentageWithin4HoursAll`.
3. No negative values in any numeric column.

In [71]:
parts_ok = (activity["NumberWithin4HoursAll"] + activity["NumberOver4HoursAll"] == 
            activity["NumberOfAttendancesAll"])
print("Rows where within + over != total : ", (~parts_ok).sum())

recalc = activity["NumberWithin4HoursAll"] / activity["NumberOfAttendancesAll"] *100
diff = (recalc - activity["PercentageWithin4HoursAll"]).abs()
print("Largest % difference : ", round(diff.max(),2))

print("Negative values: ", (activity.select_dtypes("number") < 0).sum().sum())


Rows where within + over != total :  0
Largest % difference :  0.05
Negative values:  1


**Result:**
1.  0 rows where the parts do not add up to the total.
2.  Largest difference is 0.05 percentage points, consistent with PHS rounding
   percentages to 1 decimal place.
3. 1 negative value found, which is investigated below.

### 7.1 Investigate the negative value
A negative count is impossible, so this is likely a data entry or recording error.
Identify which column and which row contain it before deciding how to handle it.

In [72]:
numeric = activity.select_dtypes("number")
print((numeric<0).sum())
activity.loc[(numeric<0).any(axis=1)]

NumberOfAttendancesAll           0
NumberWithin4HoursAll            0
NumberOver4HoursAll              1
PercentageWithin4HoursAll        0
NumberOfAttendancesEpisode       0
NumberWithin4HoursEpisode        0
NumberOver4HoursEpisode          0
PercentageWithin4HoursEpisode    0
NumberOver8HoursEpisode          0
PercentageOver8HoursEpisode      0
NumberOver12HoursEpisode         0
PercentageOver12HoursEpisode     0
dtype: int64


,Month,HBT,TreatmentLocation,DepartmentType,NumberOfAttendancesAll,NumberWithin4HoursAll,NumberOver4HoursAll,PercentageWithin4HoursAll,NumberOfAttendancesEpisode,NumberWithin4HoursEpisode,NumberOver4HoursEpisode,PercentageWithin4HoursEpisode,NumberOver8HoursEpisode,PercentageOver8HoursEpisode,NumberOver12HoursEpisode,PercentageOver12HoursEpisode,HBName,HospitalName
6280,2025-05-01,S08000028,W106H,Type 3,69,96,-27,139.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NHS Western Isles,St Brendans Cot Hospital


**Result:** 1 row affected: May 2025, NHS Western Isles, site W106H (Type 3). It reports
69 attendances but 96 seen within 4 hours, giving -27 over 4 hours (and a 4-hour
performance of ~139%). This is an error in the source data.

**Decision:** keep the row so attendance totals match published figures, but set the
three 4-hour fields to missing so this row is excluded from performance calculations.

In [73]:
bad_row = activity["NumberOver4HoursAll"] < 0
wait_cols =  ["NumberWithin4HoursAll", "NumberOver4HoursAll", "PercentageWithin4HoursAll"]
activity[wait_cols] = activity[wait_cols].mask(bad_row, axis=0)
print("Negative values:", (activity.select_dtypes("number") < 0).sum().sum())
activity.loc[bad_row, ["Month", "TreatmentLocation", "NumberOfAttendancesAll"] + wait_cols]


Negative values: 0


,Month,TreatmentLocation,NumberOfAttendancesAll,NumberWithin4HoursAll,NumberOver4HoursAll,PercentageWithin4HoursAll
6280,2025-05-01,W106H,69,NaN,NaN,NaN


**Check:** 0 negative values remain; the row keeps its 69 attendances with the 4-hour fields set to missing. 

## 8. Save cleaned data
Save both cleaned tables to `data/processed/` as CSV for use in SQL and Power BI,
then reload them to confirm the files were written correctly.

In [74]:
PROCESSED.mkdir(parents=True, exist_ok=True)

activity.to_csv(PROCESSED / "ae_activity_clean.csv", index=False)
when.to_csv(PROCESSED / "ae_when_clean.csv", index=False)

check_activity = pd.read_csv(PROCESSED / "ae_activity_clean.csv")
check_when = pd.read_csv(PROCESSED / "ae_when_clean.csv")
print("Saved activity:", check_activity.shape, "| Saved when:", check_when.shape)

Saved activity: (7211, 18) | Saved when: (640703, 13)


## Summary
Cleaned data saved to `data/processed/`:
- `ae_activity_clean.csv`: 7,211 rows × 18 columns (monthly attendances and waiting times, Jan 2018 – Aug 2026)
- `ae_when_clean.csv`: 640,703 rows × 13 columns (attendances by day of week and hour)

Key cleaning decisions:
- Filtered to January 2018 onwards and `AttendanceCategory == "All"` to avoid double counting.
- Dropped single-value and quality flag columns.
- Added health board and hospital names via left joins; 9 unmatched minor injury sites
  (0.32% of attendances) labelled "Unlisted site (code)".
- One source data error (negative over-4-hour count, NHS Western Isles, May 2025)
  had its 4-hour fields set to missing.
- Processed CSVs are not committed; run this notebook to regenerate them.